In [0]:
"""
sim_stock_delta.py
==================
Simulation script: generates synthetic Stock data and writes it into
Databricks Delta tables under  smart_odoo.silver

Tables covered (9 total):
    1. silver.stock_warehouse
    2. silver.stock_location
    3. silver.stock_lot
    4. silver.stock_move_line
    5. silver.stock_picking
    6. silver.stock_quant
    7. silver.stock_move
    8. silver.stock_warehouse_orderpoint
    9. silver.stock_scrap

════════════════════════════════════════════════════════════
SIMULATION RULES
════════════════════════════════════════════════════════════

[GENERAL]
- Catalog / Schema     : smart_odoo.silver
- Source tag           : dwh_source_db = "python_sim"
- Date range           : 2025-01-01  →  2026-04-22
- All IDs start from   : MAX(existing_id) + 1  →  safe for any re-run
- No duplicates        : every run appends brand-new rows with fresh IDs
- product_id           : 1 – 100
- partner_id           : 1 – 50
- company_id           : always 1  ("Smart Odoo LLC")

[WAREHOUSES]  silver.stock_warehouse
- Fixed 10 Egypt-based warehouses (Cairo, Alex, Giza …)
- Written only ONCE on first run (MERGE / skip if code already exists)
- reception_steps / delivery_steps : one_step | two_steps | three_steps

[LOCATIONS]   silver.stock_location
- Each warehouse gets 3 – 6 internal locations per run
- usage        : internal | transit | view (weighted: 85 % internal)
- barcode      : LOC-XXXXXX  (random 6-digit, nullable 20 %)
- replenish_location : True for ~30 % of locations
- last / next inventory dates: random within date range (nullable 40 %)

[LOTS]        silver.stock_lot
- 50 lots per run, product_id 1–100
- lot_name     : LOT-YYYY-NNNN
- avg_cost     : 5.0 – 300.0

[PICKINGS]    silver.stock_picking   (200 per run)
- state        : draft | waiting | confirmed | assigned | done | cancel
  Weights      : 5 % | 10 % | 15 % | 25 % | 35 % | 10 %
- move_type    : direct | one | two_steps
- priority     : 0 (normal) | 1 (urgent) | 2 (very urgent) – weighted 70/20/10
- date_done    : only when state = "done"
- backorder_id : nullable, references an earlier picking (~15 % chance)
- is_locked    : True when done
- picking_type : 1=Receipts | 2=Delivery | 3=Internal | 4=Returns

[STOCK MOVES] silver.stock_move   (1–4 per picking)
- state mirrors its picking
- procure_method : make_to_stock | make_to_order (80/20)
- is_in / is_out : derived from picking_type (receipt=in, delivery=out)
- is_dropship    : True ~5 %
- price_unit     : 5.0 – 500.0
- value          : price_unit × quantity
- picked         : True when state = "done"
- date_deadline  : date + 1–10 days (nullable 30 %)

[MOVE LINES]  silver.stock_move_line  (1 per move)
- quantity mirrors move quantity when done, else 0
- picked / is_entire_pack mirrors parent move

[STOCK QUANT] silver.stock_quant  (each product × 3–8 locations per run)
- quantity             : 5 – 200
- reserved_quantity    : 5–30 % of quantity
- inventory_diff       : ± 5 of quantity
- lot_id               : nullable (~40 % have a lot)

[LOTS]        silver.stock_lot
- 50 lots per run

[ORDERPOINTS] silver.stock_warehouse_orderpoint  (1 per warehouse per run)
- trigger   : orderpoint | manual
- min_qty   : 5 – 50
- max_qty   : min_qty + 50 – 200

[SCRAP]       silver.stock_scrap  (~20 per run)
- state     : draft | done   (30/70)
- scrap_qty : 1 – 20
- date_done : only when state = "done"
- should_replenish : True ~50 %

════════════════════════════════════════════════════════════
"""

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType
)

# ─────────────────────────────────────────────────────────────
# SPARK + CATALOG
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_stock_delta").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ─────────────────────────────────────────────────────────────
# REFERENCE LOOKUPS  (in-memory)
# ─────────────────────────────────────────────────────────────
PRODUCTS    = {i: f"Product_{i:03d}"   for i in range(1, 101)}   # 1-100
PARTNERS    = {i: f"Supplier_{i:03d}"  for i in range(1, 51)}    # 1-50
USERS       = {i: f"User_{i}"          for i in range(1, 11)}
UOMS        = {1: "Units", 2: "kg", 3: "liters", 4: "pcs", 5: "boxes"}
ROUTES      = {1: "Buy", 2: "Manufacture", 3: "Resupply"}

PICKING_TYPES = {
    1: "Receipts",
    2: "Delivery Orders",
    3: "Internal Transfers",
    4: "Returns",
}

WH_CITIES = [
    ("Cairo WH1",    "CAI-01"),
    ("Cairo WH2",    "CAI-02"),
    ("Cairo WH3",    "CAI-03"),
    ("Alex WH1",     "ALX-01"),
    ("Alex WH2",     "ALX-02"),
    ("Giza WH",      "GZ-01"),
    ("Delta WH",     "DLT-01"),
    ("Mansoura WH",  "MNS-01"),
    ("Tanta WH",     "TNT-01"),
    ("Aswan WH",     "ASN-01"),
]

RECEPTION_STEPS = ["one_step", "two_steps", "three_steps"]
DELIVERY_STEPS  = ["one_step", "two_steps", "three_steps"]

# ─────────────────────────────────────────────────────────────
# DATE HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 4, 22)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def maybe_date(prob_none: float = 0.3) -> datetime | None:
    return None if random.random() < prob_none else rand_date()

def maybe_int(val: int, prob_none: float = 0.2) -> int | None:
    return None if random.random() < prob_none else val

# ─────────────────────────────────────────────────────────────
# RESOLVE STARTING IDs  (append-safe – same pattern as Purchase)
# ─────────────────────────────────────────────────────────────
def max_id(table: str, col: str) -> int:
    try:
        return spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0][0]
    except Exception:
        return 0

wh_id_start       = max_id("stock_warehouse",            "warehouse_id")                  + 1
loc_id_start      = max_id("stock_location",             "stock_location_id")             + 1
lot_id_start      = max_id("stock_lot",                  "stock_lot_id")                  + 1
pick_id_start     = max_id("stock_picking",              "stock_picking_id")              + 1
move_id_start     = max_id("stock_move",                 "stock_move_id")                 + 1
ml_id_start       = max_id("stock_move_line",            "stock_move_line_id")            + 1
quant_id_start    = max_id("stock_quant",                "stock_quant_id")                + 1
op_id_start       = max_id("stock_warehouse_orderpoint", "stock_warehouse_orderpoint_id") + 1
scrap_id_start    = max_id("stock_scrap",                "stock_scrap_id")                + 1

print(f"warehouse_id start          : {wh_id_start}")
print(f"stock_location_id start     : {loc_id_start}")
print(f"stock_lot_id start          : {lot_id_start}")
print(f"stock_picking_id start      : {pick_id_start}")
print(f"stock_move_id start         : {move_id_start}")
print(f"stock_move_line_id start    : {ml_id_start}")
print(f"stock_quant_id start        : {quant_id_start}")
print(f"orderpoint_id start         : {op_id_start}")
print(f"stock_scrap_id start        : {scrap_id_start}")

now = datetime.now()


# ─────────────────────────────────────────────────────────────
# 1. WAREHOUSES  (fixed 10; skip existing codes via MERGE logic)
#    On first run → all 10 inserted.
#    On subsequent runs → nothing inserted (IDs already exist).
# ─────────────────────────────────────────────────────────────
wh_rows = []
wh_id   = wh_id_start

# Collect existing WH codes to skip duplicates
try:
    existing_codes = {
        r.code for r in
        spark.sql(f"SELECT code FROM {CATALOG}.{SCHEMA}.stock_warehouse").collect()
    }
except Exception:
    existing_codes = set()

for wh_name, code in WH_CITIES:
    if code in existing_codes:
        continue                # already loaded → skip
    wh_rows.append(Row(
        warehouse_id       = wh_id,
        company_id         = COMPANY_ID,
        company_name       = COMPANY,
        partner_id         = random.randint(1, 50),
        partner_name       = PARTNERS[random.randint(1, 50)],
        warehouse_name     = wh_name,
        code               = code,
        reception_steps    = random.choice(RECEPTION_STEPS),
        delivery_steps     = random.choice(DELIVERY_STEPS),
        active             = True,
        created_at         = now,
        updated_at         = now,
        dwh_loaded_at      = now,
        dwh_source_db      = SOURCE_DB,
    ))
    wh_id += 1


# ─────────────────────────────────────────────────────────────
# Load full warehouse list (including pre-existing) for FK refs
# ─────────────────────────────────────────────────────────────
try:
    all_wh_ids = [
        r.warehouse_id for r in
        spark.sql(f"SELECT warehouse_id FROM {CATALOG}.{SCHEMA}.stock_warehouse").collect()
    ]
except Exception:
    all_wh_ids = []

# Add any warehouses we just built (not yet committed)
for r in wh_rows:
    all_wh_ids.append(r.warehouse_id)

if not all_wh_ids:
    # Fallback: use sequential IDs for 10 warehouses
    all_wh_ids = list(range(wh_id_start, wh_id_start + 10))


# ─────────────────────────────────────────────────────────────
# 2. LOCATIONS  (3–6 per warehouse per run)
# ─────────────────────────────────────────────────────────────
loc_rows = []
loc_id   = loc_id_start

# (wh_id → list of loc_ids) built this run, used later by picks/moves/quants
wh_to_locs: dict[int, list[int]] = {wh: [] for wh in all_wh_ids}

USAGE_POOL = (["internal"] * 85) + (["transit"] * 10) + (["view"] * 5)

for wh in all_wh_ids:
    for j in range(random.randint(3, 6)):
        loc_name = f"LOC-{wh}-{j}"
        barcode  = f"LOC-{random.randint(100000,999999)}" if random.random() > 0.2 else None
        inv_date = maybe_date(0.4)
        next_inv = (inv_date + timedelta(days=random.randint(30, 180))) if inv_date else None

        loc_rows.append(Row(
            stock_location_id    = loc_id,
            company_id           = COMPANY_ID,
            company_name         = COMPANY,
            location_id          = loc_id,        # parent (self-ref simplified)
            location_name        = loc_name,
            warehouse_id         = wh,
            stock_location_name  = loc_name,
            complete_name        = f"WH{wh}/{loc_name}",
            usage                = random.choice(USAGE_POOL),
            parent_path           = f"/{wh}/{loc_id}/",
            barcode              = barcode,
            active               = True,
            replenish_location   = random.random() < 0.3,
            last_inventory_date  = inv_date,
            next_inventory_date  = next_inv,
            created_at           = now,
            updated_at           = now,
            dwh_loaded_at        = now,
            dwh_source_db        = SOURCE_DB,
        ))
        wh_to_locs[wh].append(loc_id)
        loc_id += 1

# Also load existing locations for FKs in pickings / quants
try:
    existing_locs = [
        (r.stock_location_id, r.warehouse_id) for r in
        spark.sql(
            f"SELECT stock_location_id, warehouse_id FROM {CATALOG}.{SCHEMA}.stock_location"
        ).collect()
    ]
    for lid, wid in existing_locs:
        if wid in wh_to_locs:
            wh_to_locs[wid].append(lid)
        else:
            wh_to_locs[wid] = [lid]
except Exception:
    pass

all_loc_ids = [lid for lids in wh_to_locs.values() for lid in lids]


# ─────────────────────────────────────────────────────────────
# 3. LOTS  (50 per run)
# ─────────────────────────────────────────────────────────────
lot_rows  = []
lot_id    = lot_id_start
lot_id_pool = []        # used later in quants / move_lines

for _ in range(50):
    prod_id   = random.randint(1, 100)
    loc_id_r  = random.choice(all_loc_ids) if all_loc_ids else lot_id

    lot_rows.append(Row(
        stock_lot_id      = lot_id,
        product_id        = prod_id,
        product_name      = PRODUCTS[prod_id],
        company_id        = COMPANY_ID,
        company_name      = COMPANY,
        location_id       = loc_id_r,
        location_name     = f"LOC-{loc_id_r}",
        lot_name          = f"LOT-{now.year}-{lot_id:04d}",
        ref               = f"REF-{random.randint(1000,9999)}",
        note              = random.choice([None, None, "Check before use", "Priority batch"]),
        lot_properties    = None,
        standard_price    = str(round(random.uniform(10, 300), 2)),
        avg_cost          = round(random.uniform(5.0, 300.0), 2),
        created_at        = now,
        updated_at        = now,
        dwh_loaded_at     = now,
        dwh_source_db     = SOURCE_DB,
    ))
    lot_id_pool.append(lot_id)
    lot_id += 1


# ─────────────────────────────────────────────────────────────
# 4 + 5 + 6.  PICKINGS  →  MOVES  →  MOVE LINES  (200 pickings)
# ─────────────────────────────────────────────────────────────
PICK_STATES  = (["draft"]*5   + ["waiting"]*10 + ["confirmed"]*15 +
                ["assigned"]*25 + ["done"]*35  + ["cancel"]*10)
MOVE_TYPES   = ["direct", "one", "two_steps"]
PRIORITIES   = (["0"]*70 + ["1"]*20 + ["2"]*10)
PROC_METHODS = (["make_to_stock"]*80 + ["make_to_order"]*20)

pick_rows = []
move_rows = []
ml_rows   = []

pick_id   = pick_id_start
move_id   = move_id_start
ml_id     = ml_id_start

# Collect picking IDs already committed for backorder references
try:
    prior_pick_ids = [
        r.stock_picking_id for r in
        spark.sql(
            f"SELECT stock_picking_id FROM {CATALOG}.{SCHEMA}.stock_picking LIMIT 500"
        ).collect()
    ]
except Exception:
    prior_pick_ids = []

new_pick_ids = []   # accumulate this run for backorder ref within same batch

for _ in range(200):

    wh           = random.choice(all_wh_ids)
    wh_locs      = wh_to_locs.get(wh, all_loc_ids)
    if len(wh_locs) < 2:
        wh_locs = all_loc_ids
    src_loc      = random.choice(wh_locs)
    dest_loc     = random.choice([l for l in wh_locs if l != src_loc] or wh_locs)

    state        = random.choice(PICK_STATES)
    pick_date    = rand_date()
    sched_date   = pick_date + timedelta(days=random.randint(0, 5))
    deadline     = sched_date + timedelta(days=random.randint(1, 14))
    date_done    = (pick_date + timedelta(days=random.randint(1, 3))) if state == "done" else None

    pt_id        = random.choice(list(PICKING_TYPES.keys()))
    user_id      = random.choice(list(USERS.keys()))
    partner_id   = random.randint(1, 50)

    # backorder ref (15 % chance, refs a prior picking)
    backorder_pool = prior_pick_ids + new_pick_ids
    backorder_id = (
        random.choice(backorder_pool)
        if backorder_pool and random.random() < 0.15
        else None
    )

    pick_name = f"PICK-{pick_id:06d}"

    pick_rows.append(Row(
        stock_picking_id    = pick_id,
        company_id          = COMPANY_ID,
        company_name        = COMPANY,
        partner_id          = partner_id,
        partner_name        = PARTNERS[partner_id],
        picking_type_id     = pt_id,
        picking_type_name   = PICKING_TYPES[pt_id],
        location_id         = src_loc,
        location_name       = f"LOC-{src_loc}",
        location_dest_id    = dest_loc,
        location_dest_name  = f"LOC-{dest_loc}",
        user_id             = user_id,
        user_name           = USERS[user_id],
        backorder_id        = backorder_id,
        return_id           = str(backorder_id) if backorder_id and random.random() < 0.1 else None,
        stock_picking_name  = pick_name,
        origin              = f"SO-{random.randint(1000,9999)}" if random.random() > 0.3 else None,
        move_type           = random.choice(MOVE_TYPES),
        state               = state,
        priority            = random.choice(PRIORITIES),
        note                = random.choice([None, None, "Fragile", "Urgent", "Cold chain"]),
        shipping_weight     = round(random.uniform(0.5, 50.0), 2),
        has_deadline_issue  = state in ("waiting", "confirmed") and random.random() < 0.2,
        printed             = state in ("assigned", "done"),
        is_locked           = state == "done",
        scheduled_date      = sched_date,
        date_deadline       = deadline,
        date_done           = date_done,
        created_at          = pick_date,
        updated_at          = pick_date,
        dwh_loaded_at       = now,
        dwh_source_db       = SOURCE_DB,
    ))
    new_pick_ids.append(pick_id)

    # ── MOVES (1–4 per picking) ──────────────────────────────
    for _ in range(random.randint(1, 4)):
        prod_id    = random.randint(1, 100)
        uom_id     = random.choice(list(UOMS.keys()))
        qty        = round(random.uniform(1.0, 50.0), 2)
        price_unit = round(random.uniform(5.0, 500.0), 2)
        is_in      = pt_id == 1              # Receipts
        is_out     = pt_id == 2              # Delivery
        picked     = state == "done"
        proc       = random.choice(PROC_METHODS)

        # optional final location
        loc_final  = random.choice(wh_locs) if random.random() > 0.5 else None

        # optional origin returned move
        ori_ret_id = (
            random.choice(new_pick_ids) if new_pick_ids and random.random() < 0.05 else None
        )

        move_rows.append(Row(
            stock_move_id               = move_id,
            company_id                  = COMPANY_ID,
            company_name                = COMPANY,
            product_id                  = prod_id,
            product_name                = PRODUCTS[prod_id],
            product_uom_id              = uom_id,
            product_uom_name            = UOMS[uom_id],
            location_id                 = src_loc,
            location_name               = f"LOC-{src_loc}",
            location_dest_id            = dest_loc,
            location_dest_name          = f"LOC-{dest_loc}",
            location_final_id           = loc_final,
            location_final_name         = f"LOC-{loc_final}" if loc_final else None,
            partner_id                  = maybe_int(partner_id, 0.3),
            partner_name                = PARTNERS[partner_id] if random.random() > 0.3 else None,
            picking_id                  = pick_id,
            picking_name                = pick_name,
            picking_type_id             = pt_id,
            picking_type_name           = PICKING_TYPES[pt_id],
            warehouse_id                = wh,
            warehouse_name              = next(
                (n for n, c in WH_CITIES if c == f"{['CAI','ALX','GZ','DLT','MNS','TNT','ASN'][wh % 7]}-0{(wh % 3)+1}"),
                f"WH-{wh}"
            ),
            origin_returned_move_id     = ori_ret_id,
            origin_returned_move_name   = f"PICK-{ori_ret_id:06d}" if ori_ret_id else None,
            priority                    = random.choice(PRIORITIES),
            state                       = state,
            origin                      = f"SO-{random.randint(1000,9999)}" if random.random() > 0.4 else None,
            reference                   = pick_name,
            procure_method              = proc,
            inventory_name              = None,
            next_serial                 = None,
            next_serial_count           = None,
            product_qty                 = qty,
            product_uom_qty             = qty,
            quantity                    = qty if state == "done" else 0.0,
            price_unit                  = price_unit,
            value                       = round(price_unit * qty, 2),
            picked                      = picked,
            propagate_cancel            = state == "cancel",
            is_inventory                = False,
            additional                  = False,
            to_refund                   = random.random() < 0.05,
            is_in                       = is_in,
            is_out                      = is_out,
            is_dropship                 = random.random() < 0.05,
            date                        = pick_date,
            date_deadline               = (pick_date + timedelta(days=random.randint(1, 10)))
                                          if random.random() > 0.3 else None,
            reservation_date            = (pick_date - timedelta(days=random.randint(0, 3)))
                                          if state in ("assigned", "done") else None,
            created_at                  = pick_date,
            updated_at                  = pick_date,
            dwh_loaded_at               = now,
            dwh_source_db               = SOURCE_DB,
        ))

        # ── MOVE LINE (1 per move) ───────────────────────────
        lot_ref = (
            random.choice(lot_id_pool) if lot_id_pool and random.random() > 0.5 else None
        )
        ml_rows.append(Row(
            stock_move_line_id      = ml_id,
            move_id                 = move_id,
            picking_id              = str(pick_id),
            company_id              = COMPANY_ID,
            company_name            = COMPANY,
            product_id              = prod_id,
            product_name            = PRODUCTS[prod_id],
            product_uom_id          = uom_id,
            product_uom_name        = UOMS[uom_id],
            location_id             = src_loc,
            location_name           = f"LOC-{src_loc}",
            location_dest_id        = dest_loc,
            location_dest_name      = f"LOC-{dest_loc}",
            lot_id                  = lot_ref,
            lot_name                = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
            state                   = state,
            quantity                = qty if state == "done" else 0.0,
            quantity_product_uom    = qty,
            picked                  = picked,
            is_entire_pack          = random.random() < 0.2,
            date                    = pick_date,
            created_at              = pick_date,
            updated_at              = pick_date,
            dwh_loaded_at           = now,
            dwh_source_db           = SOURCE_DB,
        ))

        move_id += 1
        ml_id   += 1

    pick_id += 1


# ─────────────────────────────────────────────────────────────
# 7. STOCK QUANT  (each product × 3–8 locations per run)
# ─────────────────────────────────────────────────────────────
quant_rows = []
quant_id   = quant_id_start

for prod_id in range(1, 101):
    sample_locs = random.sample(all_loc_ids, min(random.randint(3, 8), len(all_loc_ids)))
    for lid in sample_locs:
        qty      = round(random.uniform(5.0, 200.0), 2)
        reserved = round(qty * random.uniform(0.05, 0.30), 2)
        inv_qty  = round(qty + random.uniform(-5.0, 5.0), 2)
        diff     = round(inv_qty - qty, 2)
        inv_date = rand_date()
        lot_ref  = random.choice(lot_id_pool) if lot_id_pool and random.random() > 0.6 else None

        quant_rows.append(Row(
            stock_quant_id          = quant_id,
            product_id              = prod_id,
            product_name            = PRODUCTS[prod_id],
            company_id              = COMPANY_ID,
            company_name            = COMPANY,
            location_id             = lid,
            location_name           = f"LOC-{lid}",
            lot_id                  = lot_ref,
            lot_name                = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
            quantity                = qty,
            reserved_quantity       = reserved,
            inventory_quantity      = inv_qty,
            inventory_diff_quantity = diff,
            inventory_quantity_set  = True,
            inventory_date          = inv_date,
            in_date                 = rand_date(),
            accounting_date         = inv_date,
            created_at              = now,
            updated_at              = now,
            dwh_loaded_at           = now,
            dwh_source_db           = SOURCE_DB,
        ))
        quant_id += 1


# ─────────────────────────────────────────────────────────────
# 8. WAREHOUSE ORDERPOINTS  (1 per warehouse per run)
# ─────────────────────────────────────────────────────────────
op_rows = []
op_id   = op_id_start

TRIGGERS = ["orderpoint", "manual"]
UOM_IDS  = list(UOMS.keys())

for wh in all_wh_ids:
    wh_locs_local = wh_to_locs.get(wh, all_loc_ids)
    lid      = random.choice(wh_locs_local) if wh_locs_local else 1
    prod_id  = random.randint(1, 100)
    uom_id   = random.choice(UOM_IDS)
    route_id = random.choice(list(ROUTES.keys()))
    min_qty  = round(random.uniform(5.0, 50.0), 2)
    max_qty  = round(min_qty + random.uniform(50.0, 200.0), 2)
    to_order = round(max(0.0, min_qty - random.uniform(0, min_qty)), 2)

    op_rows.append(Row(
        stock_warehouse_orderpoint_id = op_id,
        warehouse_id                  = wh,
        warehouse_name                = f"WH-{wh}",
        location_id                   = lid,
        location_name                 = f"LOC-{lid}",
        product_id                    = prod_id,
        product_name                  = PRODUCTS[prod_id],
        company_id                    = COMPANY_ID,
        company_name                  = COMPANY,
        replenishment_uom_id          = uom_id,
        replenishment_uom_name        = UOMS[uom_id],
        route_id                      = route_id,
        route_name                    = ROUTES[route_id],
        name                          = f"OP-{wh}-{op_id:04d}",
        trigger                       = random.choice(TRIGGERS),
        product_min_qty               = min_qty,
        product_max_qty               = max_qty,
        qty_to_order_computed         = to_order,
        qty_to_order_manual           = 0.0,
        active                        = True,
        snoozed_until                 = maybe_date(0.8),
        deadline_date                 = maybe_date(0.5),
        created_at                    = now,
        updated_at                    = now,
        dwh_loaded_at                 = now,
        dwh_source_db                 = SOURCE_DB,
    ))
    op_id += 1


# ─────────────────────────────────────────────────────────────
# 9. SCRAP  (~20 per run)
# ─────────────────────────────────────────────────────────────
scrap_rows = []
scrap_id   = scrap_id_start

SCRAP_STATES = (["draft"] * 30) + (["done"] * 70)

for _ in range(20):
    prod_id  = random.randint(1, 100)
    uom_id   = random.choice(list(UOMS.keys()))
    state    = random.choice(SCRAP_STATES)
    loc_id_s = random.choice(all_loc_ids) if all_loc_ids else 1
    lot_ref  = random.choice(lot_id_pool) if lot_id_pool and random.random() > 0.5 else None
    pick_ref = random.choice(new_pick_ids) if new_pick_ids and random.random() > 0.6 else None

    scrap_rows.append(Row(
        stock_scrap_id       = scrap_id,
        company_id           = COMPANY_ID,
        company_name         = COMPANY,
        product_id           = prod_id,
        product_name         = PRODUCTS[prod_id],
        product_uom_id       = uom_id,
        product_uom_name     = UOMS[uom_id],
        lot_id               = lot_ref,
        lot_name             = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
        location_id          = loc_id_s,
        location_name        = f"LOC-{loc_id_s}",
        scrap_location_id    = random.choice(all_loc_ids) if all_loc_ids else 1,
        scrap_location_name  = "Virtual Locations/Scrap",
        picking_id           = pick_ref,
        picking_name         = f"PICK-{pick_ref:06d}" if pick_ref else None,
        name                 = f"SCRAP-{scrap_id:05d}",
        origin               = f"SO-{random.randint(1000,9999)}" if random.random() > 0.5 else None,
        state                = state,
        scrap_qty            = round(random.uniform(1.0, 20.0), 2),
        should_replenish     = random.random() < 0.5,
        date_done            = rand_date() if state == "done" else None,
        created_at           = now,
        updated_at           = now,
        dwh_loaded_at        = now,
        dwh_source_db        = SOURCE_DB,
    ))
    scrap_id += 1


# ─────────────────────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────────────────────

wh_schema = StructType([
    StructField("warehouse_id",       IntegerType(),   True),
    StructField("company_id",         IntegerType(),   True),
    StructField("company_name",       StringType(),    True),
    StructField("partner_id",         IntegerType(),   True),
    StructField("partner_name",       StringType(),    True),
    StructField("warehouse_name",     StringType(),    True),
    StructField("code",               StringType(),    True),
    StructField("reception_steps",    StringType(),    True),
    StructField("delivery_steps",     StringType(),    True),
    StructField("active",             BooleanType(),   True),
    StructField("created_at",         TimestampType(), True),
    StructField("updated_at",         TimestampType(), True),
    StructField("dwh_loaded_at",      TimestampType(), True),
    StructField("dwh_source_db",      StringType(),    True),
])

loc_schema = StructType([
    StructField("stock_location_id",   IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("location_id",         IntegerType(),   True),
    StructField("location_name",       StringType(),    True),
    StructField("warehouse_id",        IntegerType(),   True),
    StructField("stock_location_name", StringType(),    True),
    StructField("complete_name",       StringType(),    True),
    StructField("usage",               StringType(),    True),
    StructField("parent_path",         StringType(),    True),
    StructField("barcode",             StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("replenish_location",  BooleanType(),   True),
    StructField("last_inventory_date", TimestampType(), True),
    StructField("next_inventory_date", TimestampType(), True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

lot_schema = StructType([
    StructField("stock_lot_id",     IntegerType(),   True),
    StructField("product_id",       IntegerType(),   True),
    StructField("product_name",     StringType(),    True),
    StructField("company_id",       IntegerType(),   True),
    StructField("company_name",     StringType(),    True),
    StructField("location_id",      IntegerType(),   True),
    StructField("location_name",    StringType(),    True),
    StructField("lot_name",         StringType(),    True),
    StructField("ref",              StringType(),    True),
    StructField("note",             StringType(),    True),
    StructField("lot_properties",   StringType(),    True),
    StructField("standard_price",   StringType(),    True),
    StructField("avg_cost",         DoubleType(),    True),
    StructField("created_at",       TimestampType(), True),
    StructField("updated_at",       TimestampType(), True),
    StructField("dwh_loaded_at",    TimestampType(), True),
    StructField("dwh_source_db",    StringType(),    True),
])

pick_schema = StructType([
    StructField("stock_picking_id",    IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("partner_id",          IntegerType(),   True),
    StructField("partner_name",        StringType(),    True),
    StructField("picking_type_id",     IntegerType(),   True),
    StructField("picking_type_name",   StringType(),    True),
    StructField("location_id",         IntegerType(),   True),
    StructField("location_name",       StringType(),    True),
    StructField("location_dest_id",    IntegerType(),   True),
    StructField("location_dest_name",  StringType(),    True),
    StructField("user_id",             IntegerType(),   True),
    StructField("user_name",           StringType(),    True),
    StructField("backorder_id",        IntegerType(),   True),
    StructField("return_id",           StringType(),    True),
    StructField("stock_picking_name",  StringType(),    True),
    StructField("origin",              StringType(),    True),
    StructField("move_type",           StringType(),    True),
    StructField("state",               StringType(),    True),
    StructField("priority",            StringType(),    True),
    StructField("note",                StringType(),    True),
    StructField("shipping_weight",     DoubleType(),    True),
    StructField("has_deadline_issue",  BooleanType(),   True),
    StructField("printed",             BooleanType(),   True),
    StructField("is_locked",           BooleanType(),   True),
    StructField("scheduled_date",      TimestampType(), True),
    StructField("date_deadline",       TimestampType(), True),
    StructField("date_done",           TimestampType(), True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

move_schema = StructType([
    StructField("stock_move_id",               IntegerType(),   True),
    StructField("company_id",                  IntegerType(),   True),
    StructField("company_name",                StringType(),    True),
    StructField("product_id",                  IntegerType(),   True),
    StructField("product_name",                StringType(),    True),
    StructField("product_uom_id",              IntegerType(),   True),
    StructField("product_uom_name",            StringType(),    True),
    StructField("location_id",                 IntegerType(),   True),
    StructField("location_name",               StringType(),    True),
    StructField("location_dest_id",            IntegerType(),   True),
    StructField("location_dest_name",          StringType(),    True),
    StructField("location_final_id",           IntegerType(),   True),
    StructField("location_final_name",         StringType(),    True),
    StructField("partner_id",                  IntegerType(),   True),
    StructField("partner_name",                StringType(),    True),
    StructField("picking_id",                  IntegerType(),   True),
    StructField("picking_name",                StringType(),    True),
    StructField("picking_type_id",             IntegerType(),   True),
    StructField("picking_type_name",           StringType(),    True),
    StructField("warehouse_id",                IntegerType(),   True),
    StructField("warehouse_name",              StringType(),    True),
    StructField("origin_returned_move_id",     IntegerType(),   True),
    StructField("origin_returned_move_name",   StringType(),    True),
    StructField("priority",                    StringType(),    True),
    StructField("state",                       StringType(),    True),
    StructField("origin",                      StringType(),    True),
    StructField("reference",                   StringType(),    True),
    StructField("procure_method",              StringType(),    True),
    StructField("inventory_name",              StringType(),    True),
    StructField("next_serial",                 StringType(),    True),
    StructField("next_serial_count",           IntegerType(),   True),
    StructField("product_qty",                 DoubleType(),    True),
    StructField("product_uom_qty",             DoubleType(),    True),
    StructField("quantity",                    DoubleType(),    True),
    StructField("price_unit",                  DoubleType(),    True),
    StructField("value",                       DoubleType(),    True),
    StructField("picked",                      BooleanType(),   True),
    StructField("propagate_cancel",            BooleanType(),   True),
    StructField("is_inventory",                BooleanType(),   True),
    StructField("additional",                  BooleanType(),   True),
    StructField("to_refund",                   BooleanType(),   True),
    StructField("is_in",                       BooleanType(),   True),
    StructField("is_out",                      BooleanType(),   True),
    StructField("is_dropship",                 BooleanType(),   True),
    StructField("date",                        TimestampType(), True),
    StructField("date_deadline",               TimestampType(), True),
    StructField("reservation_date",            TimestampType(), True),
    StructField("created_at",                  TimestampType(), True),
    StructField("updated_at",                  TimestampType(), True),
    StructField("dwh_loaded_at",               TimestampType(), True),
    StructField("dwh_source_db",               StringType(),    True),
])

ml_schema = StructType([
    StructField("stock_move_line_id",    IntegerType(),   True),
    StructField("move_id",               IntegerType(),   True),
    StructField("picking_id",            StringType(),    True),
    StructField("company_id",            IntegerType(),   True),
    StructField("company_name",          StringType(),    True),
    StructField("product_id",            IntegerType(),   True),
    StructField("product_name",          StringType(),    True),
    StructField("product_uom_id",        IntegerType(),   True),
    StructField("product_uom_name",      StringType(),    True),
    StructField("location_id",           IntegerType(),   True),
    StructField("location_name",         StringType(),    True),
    StructField("location_dest_id",      IntegerType(),   True),
    StructField("location_dest_name",    StringType(),    True),
    StructField("lot_id",                IntegerType(),   True),
    StructField("lot_name",              StringType(),    True),
    StructField("state",                 StringType(),    True),
    StructField("quantity",              DoubleType(),    True),
    StructField("quantity_product_uom",  DoubleType(),    True),
    StructField("picked",                BooleanType(),   True),
    StructField("is_entire_pack",        BooleanType(),   True),
    StructField("date",                  TimestampType(), True),
    StructField("created_at",            TimestampType(), True),
    StructField("updated_at",            TimestampType(), True),
    StructField("dwh_loaded_at",         TimestampType(), True),
    StructField("dwh_source_db",         StringType(),    True),
])

quant_schema = StructType([
    StructField("stock_quant_id",           IntegerType(),   True),
    StructField("product_id",               IntegerType(),   True),
    StructField("product_name",             StringType(),    True),
    StructField("company_id",               IntegerType(),   True),
    StructField("company_name",             StringType(),    True),
    StructField("location_id",              IntegerType(),   True),
    StructField("location_name",            StringType(),    True),
    StructField("lot_id",                   IntegerType(),   True),
    StructField("lot_name",                 StringType(),    True),
    StructField("quantity",                 DoubleType(),    True),
    StructField("reserved_quantity",        DoubleType(),    True),
    StructField("inventory_quantity",       DoubleType(),    True),
    StructField("inventory_diff_quantity",  DoubleType(),    True),
    StructField("inventory_quantity_set",   BooleanType(),   True),
    StructField("inventory_date",           TimestampType(), True),
    StructField("in_date",                  TimestampType(), True),
    StructField("accounting_date",          TimestampType(), True),
    StructField("created_at",               TimestampType(), True),
    StructField("updated_at",               TimestampType(), True),
    StructField("dwh_loaded_at",            TimestampType(), True),
    StructField("dwh_source_db",            StringType(),    True),
])

op_schema = StructType([
    StructField("stock_warehouse_orderpoint_id", IntegerType(),   True),
    StructField("warehouse_id",                  IntegerType(),   True),
    StructField("warehouse_name",                StringType(),    True),
    StructField("location_id",                   IntegerType(),   True),
    StructField("location_name",                 StringType(),    True),
    StructField("product_id",                    IntegerType(),   True),
    StructField("product_name",                  StringType(),    True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("replenishment_uom_id",          IntegerType(),   True),
    StructField("replenishment_uom_name",        StringType(),    True),
    StructField("route_id",                      IntegerType(),   True),
    StructField("route_name",                    StringType(),    True),
    StructField("name",                          StringType(),    True),
    StructField("trigger",                       StringType(),    True),
    StructField("product_min_qty",               DoubleType(),    True),
    StructField("product_max_qty",               DoubleType(),    True),
    StructField("qty_to_order_computed",         DoubleType(),    True),
    StructField("qty_to_order_manual",           DoubleType(),    True),
    StructField("active",                        BooleanType(),   True),
    StructField("snoozed_until",                 TimestampType(), True),
    StructField("deadline_date",                 TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])

scrap_schema = StructType([
    StructField("stock_scrap_id",       IntegerType(),   True),
    StructField("company_id",           IntegerType(),   True),
    StructField("company_name",         StringType(),    True),
    StructField("product_id",           IntegerType(),   True),
    StructField("product_name",         StringType(),    True),
    StructField("product_uom_id",       IntegerType(),   True),
    StructField("product_uom_name",     StringType(),    True),
    StructField("lot_id",               IntegerType(),   True),
    StructField("lot_name",             StringType(),    True),
    StructField("location_id",          IntegerType(),   True),
    StructField("location_name",        StringType(),    True),
    StructField("scrap_location_id",    IntegerType(),   True),
    StructField("scrap_location_name",  StringType(),    True),
    StructField("picking_id",           IntegerType(),   True),
    StructField("picking_name",         StringType(),    True),
    StructField("name",                 StringType(),    True),
    StructField("origin",               StringType(),    True),
    StructField("state",                StringType(),    True),
    StructField("scrap_qty",            DoubleType(),    True),
    StructField("should_replenish",     BooleanType(),   True),
    StructField("date_done",            TimestampType(), True),
    StructField("created_at",           TimestampType(), True),
    StructField("updated_at",           TimestampType(), True),
    StructField("dwh_loaded_at",        TimestampType(), True),
    StructField("dwh_source_db",        StringType(),    True),
])


# ─────────────────────────────────────────────────────────────
# CREATE DATAFRAMES
# ─────────────────────────────────────────────────────────────
df_wh    = spark.createDataFrame(wh_rows,    schema=wh_schema)
df_loc   = spark.createDataFrame(loc_rows,   schema=loc_schema)
df_lot   = spark.createDataFrame(lot_rows,   schema=lot_schema)
df_pick  = spark.createDataFrame(pick_rows,  schema=pick_schema)
df_move  = spark.createDataFrame(move_rows,  schema=move_schema)
df_ml    = spark.createDataFrame(ml_rows,    schema=ml_schema)
df_quant = spark.createDataFrame(quant_rows, schema=quant_schema)
df_op    = spark.createDataFrame(op_rows,    schema=op_schema)
df_scrap = spark.createDataFrame(scrap_rows, schema=scrap_schema)

print(f"stock_warehouse rows             : {df_wh.count()}")
print(f"stock_location rows              : {df_loc.count()}")
print(f"stock_lot rows                   : {df_lot.count()}")
print(f"stock_picking rows               : {df_pick.count()}")
print(f"stock_move rows                  : {df_move.count()}")
print(f"stock_move_line rows             : {df_ml.count()}")
print(f"stock_quant rows                 : {df_quant.count()}")
print(f"stock_warehouse_orderpoint rows  : {df_op.count()}")
print(f"stock_scrap rows                 : {df_scrap.count()}")


# ─────────────────────────────────────────────────────────────
# WRITE TO DELTA  (append – no duplicates guaranteed by fresh IDs)
# ─────────────────────────────────────────────────────────────
def append_delta(df, table: str):
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    )
    print(f"  ✅  {table}")

append_delta(df_wh,    "stock_warehouse")
append_delta(df_loc,   "stock_location")
append_delta(df_lot,   "stock_lot")
append_delta(df_pick,  "stock_picking")
append_delta(df_move,  "stock_move")
append_delta(df_ml,    "stock_move_line")
append_delta(df_quant, "stock_quant")
append_delta(df_op,    "stock_warehouse_orderpoint")
append_delta(df_scrap, "stock_scrap")

print("\n✅  DONE — All 9 stock Delta tables written to smart_odoo.silver")